# K-Nearest-Neighbors
KNN falls in the supervised learning family of algorithms. Informally, this means that we are given a labelled dataset consiting 
of training observations (x,y) and would like to capture the relationship between x and y. More formally, our goal is to learn a
function h:X→Y so that given an unseen observation x, h(x) can confidently predict the corresponding output y.

In this module we will explore the inner workings of KNN, choosing the optimal K values and using KNN from scikit-learn.

# Overview

1.Read the problem statement.

2.Get the dataset.

3.Explore the dataset.

4.Pre-processing of dataset.

5.Visualization

6.Transform the dataset for building machine learning model.

7.Split data into train, test set.

7.Build Model.

8.Apply the model.

9.Evaluate the model.

10.Finding Optimal K value

11.Repeat 7,8,9 steps.

# Problem statement

Dataset:
The data set we’ll be using is the Iris Flower Dataset which was first introduced in 1936 by the famous statistician Ronald 
Fisher and consists of 50 observations from each of three species of Iris (Iris setosa, Iris virginica and Iris versicolor). 
Four features were measured from each sample: the length and the width of the sepals and petals.

Source: https://archive.ics.uci.edu/ml/datasets/Iris

Train the KNN algorithm to be able to distinguish the species from one another given the measurements of the 4 features.

# Question 1
Import libraries and load the dataset

In [20]:
import pandas as pd
import numpy as np

# Pre processing 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder



#visualisation
import plotly.express as plt
import seaborn as sns

#Classification Model
from sklearn.model_selection import train_test_split

In [2]:
#loading the IFRIS data

iris_data = pd.read_csv('iris-1.csv',skiprows=0)
iris_data.info()
iris_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             152 non-null    int64  
 1   SepalLengthCm  151 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   151 non-null    float64
 5   Species        152 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 7.2+ KB


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
#Checking if the data has any nulls
iris_data.isna().sum()

Id               0
SepalLengthCm    1
SepalWidthCm     2
PetalLengthCm    2
PetalWidthCm     1
Species          0
dtype: int64

Looks like "SepalLengthCm , SepalWidthCm PetalLengthCm, PetalWidthCm" columns have MISSING values. 

# Question 2 - Estimating missing values
Its not good to remove the records having missing values all the time. We may end up loosing some data points. So, we will have 
to see how to replace those missing values with some estimated values (median)

In [4]:
#copying the Iris data columns for post transform activity
iris_data = iris_data.fillna(iris_data.median())
iris_data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


# Question 3 - Dealing with categorical data
Change all the classes to numericals (0to2).

In [5]:
iris_data.iloc[:,-1].unique()

array(['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'], dtype=object)

In [6]:
#Replacing categorical variables to numbers
from sklearn.preprocessing import LabelEncoder
labelencode = LabelEncoder()
iris_data.iloc[:,-1] = labelencode.fit_transform(iris_data.iloc[:,-1])

iris_data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0


# Question 4
Observe the association of each independent variable with target variable and drop variables from feature set having correlation 
in range -0.1 to 0.1 with target variable.

In [7]:
iris_data.corr()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
Id,1.000000,0.702734,-0.392693,0.872346,0.890676,0.942753
SepalLengthCm,0.702734,1.000000,-0.109369,0.871120,0.815986,0.775061
SepalWidthCm,-0.392693,-0.109369,1.000000,-0.420713,-0.356510,-0.417318
PetalLengthCm,0.872346,0.871120,-0.420713,1.000000,0.962043,0.944477
PetalWidthCm,0.890676,0.815986,-0.356510,0.962043,1.000000,0.952513
Species,0.942753,0.775061,-0.417318,0.944477,0.952513,1.000000


In [8]:



fig = plt.imshow(iris_data.corr(), x = iris_data.columns, y = iris_data.columns)
fig.show()

# Question 5
Observe the independent variables variance and drop such variables having no variance or almost zero variance(variance < 0.1). 
They will be having almost no influence on the classification.

In [11]:
iris_data.var()

Id               1938.000000
SepalLengthCm       0.676645
SepalWidthCm        0.185552
PetalLengthCm       3.076516
PetalWidthCm        0.577141
Species             0.675322
dtype: float64

In [86]:
fig = plt.box(iris_data.iloc[:,1:5], points ="all")
fig.show()

In [18]:
fig = plt.scatter_matrix(iris_data.iloc[:,1:6], color="Species", width=1000, height=1000)
fig.show()

# Question 6
Plot the scatter matrix for all the variables.

In [90]:

fig = plt.scatter_matrix(iris_data.iloc[:,1:6],color = "Species",width=90, height=100)
fig.show()

# Split the dataset into training and test sets

# Question 7
Split the dataset into training and test sets with 80-20 ratio.

In [24]:
x_train, x_test , y_train, y_test = train_test_split(iris_data.iloc[:,1:5], iris_data.iloc[:,-1],random_state=42,test_size=0.3)
x_train.info()
y_train.count()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 106 entries, 110 to 102
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   SepalLengthCm  106 non-null    float64
 1   SepalWidthCm   106 non-null    float64
 2   PetalLengthCm  106 non-null    float64
 3   PetalWidthCm   106 non-null    float64
dtypes: float64(4)
memory usage: 4.1 KB


106

# Question 8 - Model
Build the model and train and test on training and test sets respectively using scikit-learn. Print the Accuracy of the model with different values of k=3,5,9.

Hint: For accuracy you can check accuracy_score() in scikit-learn

In [33]:
for i in [3,5,9]:
 print (i)


3
5
9


In [43]:
for i in [3,5,9]:
    model = KNeighborsClassifier(n_neighbors=i)
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    print("Accuracy Score for Model wih {} Neigbhours: {}".format(i,accuracy_score(y_test,y_pred)))

Accuracy Score for Model wih 3 Neigbhours: 1.0
Accuracy Score for Model wih 5 Neigbhours: 0.9782608695652174
Accuracy Score for Model wih 9 Neigbhours: 0.9565217391304348


In [25]:
# loading library
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score



In [52]:
n_count = list(np.arange(1,20,2))n_count = list(np.arange(1,20,2))

[1, 3, 5, 7, 9, 11, 13, 15, 17, 19]

# Question 9 - Cross Validation
Run the KNN with no of neighbours to be 1,3,5..19 and *Find the optimal number of neighbours from the above list using the Mis 
classification error

Hint:

Misclassification error (MSE) = 1 - Test accuracy score. Calculated MSE for each model with neighbours = 1,3,5...19 and find the 
model with lowest MSE

In [56]:

# Question 9 - Cross Validation
score = []

for i in n_count:
    model = KNeighborsClassifier(n_neighbors=i)
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    score.append(accuracy_score(y_test,y_pred))

MSE = [1 - x for x in score]


[0.021739130434782594,
 0.0,
 0.021739130434782594,
 0.021739130434782594,
 0.04347826086956519,
 0.021739130434782594,
 0.021739130434782594,
 0.021739130434782594,
 0.021739130434782594,
 0.021739130434782594]

# Question 10
Plot misclassification error vs k (with k value on X-axis) using matplotlib.

In [76]:
fig = plt.line(title= "Misclassification", y=MSE,x=n_count)
fig.update_xaxes(tickangle=45, tickfont=dict(family='Rockwell', color='crimson', size=14),title="Neighbour count")
fig.update_yaxes(title="Error")
fig.show()